# 1 Gen Proposals

This notebook runs the redesigned proposal-generation stage for the study.

It does exactly two things:
1. Pass A: generate AI ideas for each requested condition.
2. Pass B: after all ideas are saved, expand those ideas into full proposals with the same model.

It does not run rephrasing, embedding preparation, or analyses.

In [ ]:
CONDITIONS_TO_RUN = ['baseline', 'one_at_a_time', 'persona']
MODELS_TO_USE = ['gpt-5.5', 'gemini-3.1-pro-preview', 'claude-sonnet-5']

GENERATION_TEMPERATURE = 0.9
MAX_TOKENS_IDEAS = 8000
MAX_TOKENS_PROPOSALS = 16000
RETRY_DELAYS = [2, 5, 10]
SAVE_PROGRESS_EVERY_N_CALLS = 3
RESUME_OK = True

RUN_TEST_CALLS = False
TEST_MODEL = MODELS_TO_USE[0]
TEST_MAX_TOKENS = MAX_TOKENS_IDEAS

# Set to a fixed string only if you want to pin new outputs to a custom run id.
RUN_ID = None


In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ai_models_interface import AIModelsInterface
from prompt_templates import PromptManager
from proposal_generation import (
    build_condition_registry,
    build_idea_prompt_kwargs,
    find_project_root,
    idea_schedule_rows,
    load_human_target_roster,
    load_persona_cards,
    load_shared_call_context,
    load_target_proposal_count,
    now_run_id,
    parse_generated_ideas_response,
    run_idea_generation_for_condition,
    run_proposal_expansion_for_condition,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
shared_call_context = load_shared_call_context(PROJECT_ROOT)
human_target_roster = load_human_target_roster(PROJECT_ROOT)
target_n = load_target_proposal_count(PROJECT_ROOT)
if target_n != 23:
    raise RuntimeError(f'Expected 23 human comparison proposals, found {target_n}.')

persona_roster = load_persona_cards(PROJECT_ROOT, human_target_roster)
if 'persona' in CONDITIONS_TO_RUN and len(persona_roster) != target_n:
    raise RuntimeError(
        f'Persona roster must contain exactly {target_n} cards, found {len(persona_roster)}.'
    )

prompt_manager = PromptManager()
ai_interface = AIModelsInterface(config_path='.env', override_env=True)
available_models = ai_interface.get_available_models()
resolved_models = [ai_interface.resolve_model_name(model_name) for model_name in MODELS_TO_USE]
missing_models = [model_name for model_name in resolved_models if model_name not in available_models]
if missing_models:
    raise RuntimeError(f'Requested model(s) unavailable with current API keys: {missing_models}')

condition_registry = build_condition_registry(target_n)
stage_run_id = RUN_ID or now_run_id()

print(f'Project root: {PROJECT_ROOT}')
print(f'Human target proposals: {target_n}')
print(f'Available canonical models: {available_models}')
print(f'Conditions to run: {CONDITIONS_TO_RUN}')


## Optional Test Calls

Set `RUN_TEST_CALLS = True` to make one sample idea-generation call per condition before the full generation passes. This lets you inspect the exact prompt, raw model output, and parsed JSON without launching all draws.


In [ ]:
test_call_outputs = {}

if not RUN_TEST_CALLS:
    print('RUN_TEST_CALLS is False; skipping sample proposal-generation calls.')
else:
    test_model = ai_interface.resolve_model_name(TEST_MODEL)
    if test_model not in resolved_models:
        raise RuntimeError(f'TEST_MODEL must be one of MODELS_TO_USE after alias resolution: {resolved_models}')

    for condition in CONDITIONS_TO_RUN:
        condition_config = condition_registry[condition]
        schedule = idea_schedule_rows(
            condition_config=condition_config,
            model_name=test_model,
            target_n=target_n,
            persona_roster=persona_roster if condition == 'persona' else None,
        )
        sample_row = schedule[0]
        prompt_kwargs = build_idea_prompt_kwargs(
            shared_call_context=shared_call_context,
            condition_config=condition_config,
            target_n=target_n,
            schedule_row=sample_row,
        )
        prompt = prompt_manager.format_prompt(condition_config['idea_prompt_template'], prompt_kwargs)
        result = ai_interface.generate_content_with_metadata(
            prompt,
            model_name=test_model,
            temperature=GENERATION_TEMPERATURE,
            max_tokens=TEST_MAX_TOKENS,
            retry_delays=RETRY_DELAYS,
        )
        parsed_ideas, parse_error = parse_generated_ideas_response(
            result['raw_response'],
            expected_count=int(sample_row['expected_idea_count']),
            mode=condition_config['idea_generation_mode'],
        ) if not result['error'] else ([], None)

        test_call_outputs[condition] = {
            'condition': condition,
            'model': test_model,
            'prompt_template': condition_config['idea_prompt_template'],
            'idea_generation_mode': condition_config['idea_generation_mode'],
            'sample_call_id': sample_row['idea_call_id'],
            'expected_idea_count': int(sample_row['expected_idea_count']),
            'prompt': prompt,
            'raw_response': result['raw_response'],
            'provider_model_id': result.get('provider_model_id', ''),
            'timestamp': result.get('timestamp', ''),
            'error': result.get('error', ''),
            'parse_error': parse_error,
            'parsed_ideas': parsed_ideas,
        }

    test_call_summary = pd.DataFrame([
        {
            'condition': payload['condition'],
            'model': payload['model'],
            'prompt_template': payload['prompt_template'],
            'expected_idea_count': payload['expected_idea_count'],
            'parsed_idea_count': len(payload['parsed_ideas']),
            'error': payload['error'],
            'parse_error': payload['parse_error'],
        }
        for payload in test_call_outputs.values()
    ])
    display(test_call_summary)

    for condition, payload in test_call_outputs.items():
        print(f'\n=== Test Call: {condition} ===')
        print('Prompt:')
        print(payload['prompt'])
        print('\nRaw response:')
        print(payload['raw_response'])
        print('\nParsed ideas:')
        display(pd.DataFrame(payload['parsed_ideas']))


In [ ]:
# Pass A: generate and save all ideas before any proposal expansion begins.
idea_stage_outputs = {}

for condition in CONDITIONS_TO_RUN:
    print(f'\n=== Pass A: {condition} ===')
    condition_result = run_idea_generation_for_condition(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        prompt_manager=prompt_manager,
        shared_call_context=shared_call_context,
        condition_config=condition_registry[condition],
        models_to_use=resolved_models,
        target_n=target_n,
        persona_roster=persona_roster if condition == 'persona' else None,
        generation_temperature=GENERATION_TEMPERATURE,
        max_tokens=MAX_TOKENS_IDEAS,
        retry_delays=RETRY_DELAYS,
        save_progress_every_n_calls=SAVE_PROGRESS_EVERY_N_CALLS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    idea_stage_outputs[condition] = condition_result
    print(f"Ideas file: {condition_result['ideas_path']}")
    print(f"Rows: {len(condition_result['ideas_df'])}")
    if condition_result['qa_issues']:
        print('QA issues:')
        for issue in condition_result['qa_issues']:
            print(f'  - {issue}')

idea_qa_summary = []
for condition, result in idea_stage_outputs.items():
    idea_df = result['ideas_df']
    duplicate_titles = int(idea_df['title'].fillna('').duplicated().sum()) if not idea_df.empty else 0
    idea_qa_summary.append(
        {
            'condition': condition,
            'rows': len(idea_df),
            'models': ', '.join(sorted(idea_df['model'].dropna().unique())) if not idea_df.empty else '',
            'duplicate_titles': duplicate_titles,
            'qa_issue_count': len(result['qa_issues']),
            'reused_existing': result['reused_existing'],
        }
    )

pd.DataFrame(idea_qa_summary)


In [ ]:
# Pass B: expand ideas into full proposals only after Pass A completes for all requested conditions.
proposal_stage_outputs = {}

for condition in CONDITIONS_TO_RUN:
    print(f'\n=== Pass B: {condition} ===')
    proposal_result = run_proposal_expansion_for_condition(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        prompt_manager=prompt_manager,
        shared_call_context=shared_call_context,
        condition_config=condition_registry[condition],
        ideas_df=idea_stage_outputs[condition]['ideas_df'],
        generation_temperature=GENERATION_TEMPERATURE,
        max_tokens=MAX_TOKENS_PROPOSALS,
        retry_delays=RETRY_DELAYS,
        save_progress_every_n_calls=SAVE_PROGRESS_EVERY_N_CALLS,
        resume_ok=RESUME_OK,
        run_id=idea_stage_outputs[condition]['run_id'],
    )
    proposal_stage_outputs[condition] = proposal_result
    print(f"Progress file: {proposal_result['progress_path']}")
    print(f"Complete file: {proposal_result['complete_path']}")
    print(f"Rows: {len(proposal_result['proposals_df'])}")
    if proposal_result['qa_issues']:
        print('Proposal QA issues:')
        for issue in proposal_result['qa_issues'][:10]:
            print(f'  - {issue}')


In [ ]:
summary_rows = []
for condition in CONDITIONS_TO_RUN:
    idea_result = idea_stage_outputs[condition]
    proposal_result = proposal_stage_outputs[condition]
    proposal_df = proposal_result['proposals_df']
    completed_rows = 0
    if not proposal_df.empty:
        completed_rows = int(
            proposal_df['background_and_significance'].fillna('').astype(str).str.strip().ne('').sum()
        )
    summary_rows.append(
        {
            'condition': condition,
            'idea_rows': len(idea_result['ideas_df']),
            'proposal_rows': len(proposal_df),
            'completed_proposals': completed_rows,
            'idea_file': str(idea_result['ideas_path']),
            'proposal_complete_file': str(proposal_result['complete_path']) if proposal_result['complete_path'] else '',
            'idea_reused_existing': idea_result['reused_existing'],
            'proposal_reused_existing': proposal_result['reused_existing'],
            'idea_qa_issue_count': len(idea_result['qa_issues']),
            'proposal_qa_issue_count': len(proposal_result['qa_issues']),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df
